<a href="https://colab.research.google.com/github/l-ordkp/Finetuned-whisper-large-v3/blob/main/fine_tuning_whisper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import re
v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
!pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install librosa soundfile evaluate jiwer torchcodec "datasets>=3.4.1,<4.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 12.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2025.11.4 req

In [3]:
from unsloth import FastModel
from transformers import WhisperForConditionalGeneration
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/whisper-large-v3",
    dtype = None, # Leave as None for auto detection
    load_in_4bit = False, # Set to True to do 4bit quantization which reduces memory
    auto_model = WhisperForConditionalGeneration,
    whisper_language = "hi",
    whisper_task = "transcribe",
)

==((====))==  Unsloth 2025.11.3: Fast Whisper patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.
unsloth/whisper-large-v3 does not have a padding token! Will use pad_token = <|endoftext|>.


In [5]:
# Colab cell (python)
from datasets import load_dataset, Audio
import random, IPython.display as ipd

print("Loading ai4bharat/IndicVoices- may take a minute on first download...")
ds = load_dataset("ai4bharat/IndicVoices", "hindi", split="train")

# show dataset size and columns
print("Dataset size:", len(ds))
print("Columns:", ds.column_names)

# ensure audio column is cast to 16k (Whisper expects 16000)
ds = ds.cast_column("audio", Audio(sampling_rate=16000))



Loading ai4bharat/IndicVoices- may take a minute on first download...


Resolving data files:   0%|          | 0/91 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/74 [00:00<?, ?it/s]

hindi/valid-00000-of-00001.parquet:   0%|          | 0.00/367M [00:00<?, ?B/s]

hindi/train-00000-of-00074.parquet:   0%|          | 0.00/393M [00:00<?, ?B/s]

...

Generating valid split:   0%|          | 0/4180 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/333256 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/71 [00:00<?, ?it/s]

Dataset size: 333256
Columns: ['audio_filepath', 'text', 'duration', 'lang', 'samples', 'verbatim', 'normalized', 'speaker_id', 'scenario', 'task_name', 'gender', 'age_group', 'job_type', 'qualification', 'area', 'district', 'state', 'occupation', 'verification_report', 'unsanitized_verbatim', 'unsanitized_normalized']
=== Example 213464 ===
Text fields available: ['text', 'lang', 'verbatim', 'normalized', 'speaker_id', 'scenario', 'task_name', 'gender', 'age_group', 'job_type', 'qualification', 'area', 'district', 'state', 'occupation', 'verification_report', 'unsanitized_verbatim', 'unsanitized_normalized']
Transcript: नीचे


TypeError: 'NoneType' object is not subscriptable

In [6]:
!pip install librosa soundfile

In [10]:
from datasets import load_dataset
import librosa
import soundfile as sf
import random
import IPython.display as ipd

for _ in range(3):
    idx = random.randint(0, len(ds)-1)
    sample = ds[idx]

    print(f"\n=== Example {idx} ===")
    transcript = sample["text"]
    print("Transcript:", transcript)

    # Load audio data directly from the 'audio' feature, which was cast to Audio(sampling_rate=16000)
    if sample["audio"] is not None:
        audio = sample["audio"]["array"]
        sr = sample["audio"]["sampling_rate"]

        print("Audio shape:", audio.shape, "Sample rate:", sr)

        # Play audio
        display(ipd.Audio(audio, rate=sr))
    else:
        print(f"Audio data for sample {idx} is missing or corrupt. Skipping.")


=== Example 293898 ===
Transcript: और ये च ये चमेली वाला कितने का बताए हैं
Audio data for sample 293898 is missing or corrupt. Skipping.

=== Example 304951 ===
Transcript: चार मार्च उन्नीस सौ अठत्तर बारह सितंबर उन्नीस सौ अस्सी आठ फरवरी उन्नीस सौ तिरासी एक फरवरी उन्नीस सौ बयासी अक्टूबर दो हजार आठ
Audio data for sample 304951 is missing or corrupt. Skipping.

=== Example 201762 ===
Transcript: वैसे अ जैसे कि मैं आपको बताने जा रहीं हूँ जो ध्यान प्राणायाम है जो हमारे जीवन में बहुत महत्वपूर्ण है जैसे कि ये आपका योग्यवास बहुत महत्वपूर्ण भूमिका निभाता है आपके जीवन में जैसे कि मैं कोई चीज अपने दिमाग में ध्यान रखती हूँ बट मैं उसको नहीं बोल पाती हूँ
Audio data for sample 201762 is missing or corrupt. Skipping.
